In [1]:
import io
import numpy as np
import zipfile
from pathlib import Path
import pandas as pd
import requests
from sqlalchemy import create_engine
import folium
import geopandas as gpd
from shapely.geometry import LineString, Point

In [2]:
DB_DIR = Path.cwd() / "data"
DB_DIR.mkdir(exist_ok=True)
DB_PATH = DB_DIR / "ctrain.db"


In [3]:
engine = create_engine(f"sqlite:///{DB_PATH}")

In [ ]:
GTFS_URL = "https://data.calgary.ca/download/npk7-z3bj/application%2Fzip"

response = requests.get(GTFS_URL)
if response.status_code != 200:
    raise Exception(
        f"Failed to fetch GTFS zip file. HTTP Status: {response.status_code}"
    )

zip_file = zipfile.ZipFile(io.BytesIO(response.content))

In [5]:
def gtfs_time_to_seconds(time_str):
    if pd.isna(time_str):
        return None
    h, m, s = map(int, str(time_str).strip().split(":"))
    return h * 3600 + m * 60 + s


In [18]:
routes_df = pd.read_csv(zip_file.open("routes.txt"), dtype=str)
ctrain_routes = routes_df[routes_df["route_short_name"].isin(["201", "202"])]
ctrain_route_ids = ctrain_routes["route_id"].unique()
ctrain_routes.to_sql("routes", engine, if_exists="replace", index=False)
ctrain_routes

,route_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
131,201-20785,201,Red Line - Somerset - Bridlewood/Tuscany CTrain,NaN,0,NaN,NaN,NaN
132,201-20797,201,Red Line - Somerset - Bridlewood/Tuscany CTrain,NaN,0,NaN,NaN,NaN
133,202-20785,202,Blue Line - Saddletowne/69 Street CTrain,NaN,0,NaN,NaN,NaN
134,202-20797,202,Blue Line - Saddletowne/69 Street CTrain,NaN,0,NaN,NaN,NaN


In [19]:
trips_df = pd.read_csv(zip_file.open("trips.txt"), dtype=str)
ctrain_trips = trips_df[trips_df["route_id"].isin(ctrain_route_ids)]
ctrain_trip_ids = ctrain_trips["trip_id"].unique()
ctrain_trips.to_sql("trips", engine, if_exists="replace", index=False)
ctrain_trips

,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id
23750,201-20785,2026JU-1LRTSA-Saturday-00,73744902,SOMERSET-BRIDLEWOOD,1,7097302,2010682
23751,201-20785,2026JU-1LRTSA-Saturday-00,73744905,TUSCANY,0,7097302,2010689
23752,201-20785,2026JU-1LRTSA-Saturday-00,73744906,TUSCANY,0,7097302,2010681
23753,201-20785,2026JU-1LRTSA-Saturday-00,73744907,SOMERSET-BRIDLEWOOD,1,7097302,2010682
23754,201-20785,2026JU-1LRTSA-Saturday-00,73744908,TUSCANY,0,7097302,2010681
...,...,...,...,...,...,...,...
29029,202-20797,2026JU-pT7Aug02-Sunday-01,75391275,69 ST STATION,1,7199254,2020440
29030,202-20797,2026JU-pT7Aug02-Sunday-01,75391276,NE CTRAIN,0,7199254,2020443
29031,202-20797,2026JU-pT7Aug02-Sunday-01,75391278,69 ST STATION,1,7199254,2020440
29032,202-20797,2026JU-pT7Aug02-Sunday-01,75391279,NE CTRAIN,0,7199254,2020443


In [24]:
stop_times_df = pd.read_csv(zip_file.open("stop_times.txt"), dtype=str)
ctrain_stop_times = stop_times_df[
    stop_times_df["trip_id"].isin(ctrain_trip_ids)
].copy()
ctrain_stop_times["arrival_sec"] = ctrain_stop_times["arrival_time"].apply(
    gtfs_time_to_seconds
)
ctrain_stop_times["departure_sec"] = ctrain_stop_times["departure_time"].apply(
    gtfs_time_to_seconds
)
ctrain_stop_times.to_sql(
    "stop_times", engine, if_exists="replace", index=False
)
ctrain_stop_times

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,shape_dist_traveled,timepoint,arrival_sec,departure_sec
830049,73744902,04:54:00,04:54:00,3641,1,0,0,0.000,1,17640,17640
830050,73744902,04:57:00,04:57:00,3817,2,0,0,2.398,1,17820,17820
830051,73744902,05:01:00,05:01:00,3960,3,0,0,6.263,1,18060,18060
830052,73744902,05:04:00,05:04:00,6815,4,0,0,9.070,1,18240,18240
830053,73744902,05:06:00,05:06:00,8566,5,0,0,10.005,1,18360,18360
...,...,...,...,...,...,...,...,...,...,...,...
912151,75391281,25:26:00,25:26:00,3634,4,0,0,3.430,1,91560,91560
912152,75391281,25:28:00,25:28:00,3632,5,0,0,4.337,1,91680,91680
912153,75391281,25:30:00,25:30:00,3630,6,0,0,5.687,1,91800,91800
912154,75391281,25:32:00,25:32:00,3628,7,0,0,6.975,1,91920,91920


In [27]:
stops_df = pd.read_csv(zip_file.open("stops.txt"), dtype=str)
ctrain_stops = stops_df[
    stops_df["stop_id"].isin(ctrain_stop_times["stop_id"].unique())
]
ctrain_stops.to_sql("stops", engine, if_exists="replace", index=False)
ctrain_stops

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type
937,3626,3626,WB 69 Street CTrain Station,NaN,51.037605,-114.188486,NaN,NaN,0
938,3627,3627,EB 69 Street CTrain Station,NaN,51.037507,-114.188475,NaN,NaN,0
939,3628,3628,WB Sirocco CTrain Station,NaN,51.038152,-114.169585,NaN,NaN,0
940,3629,3629,EB Sirocco CTrain Station,NaN,51.038105,-114.169484,NaN,NaN,0
941,3630,3630,WB 45 Street CTrain Station,NaN,51.037942,-114.153369,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
5472,9396,9396,NB Martindale CTrain Station,NaN,51.117409,-113.967973,NaN,NaN,0
5473,9398,9398,NB Saddletowne CTrain Station,NaN,51.125542,-113.948277,NaN,NaN,0
5782,9781,9781,SB Saddletowne CTrain Station,NaN,51.125895,-113.948358,NaN,NaN,0
5841,9896,9896,NB McKnight - Westwinds CTrain Station,NaN,51.109530,-113.975160,NaN,NaN,0


In [28]:
shapes_df = pd.read_csv(zip_file.open("shapes.txt"), dtype=str)
ctrain_shapes = shapes_df[
    shapes_df["shape_id"].isin(ctrain_trips["shape_id"].dropna().unique())
]
ctrain_shapes.to_sql("shapes", engine, if_exists="replace", index=False)
ctrain_shapes

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
175690,2010486,50.898750,-114.068731,10001,0.000
175691,2010486,50.898907,-114.068816,10002,0.018
175692,2010486,50.899806,-114.069289,10003,0.124
175693,2010486,50.900114,-114.069440,10004,0.160
175694,2010486,50.900633,-114.069679,10005,0.220
...,...,...,...,...,...
181872,2020443,51.047104,-114.083917,70007,7.947
181873,2020443,51.047080,-114.083076,70008,8.006
181874,2020443,51.047076,-114.082860,70009,8.021
181875,2020443,51.047074,-114.082843,70010,8.023


In [30]:
# Check station names in the database
inspect_stops_query = """
SELECT DISTINCT stop_id, stop_name, stop_lat, stop_lon
FROM stops
ORDER BY stop_name;
"""

df_stops = pd.read_sql_query(inspect_stops_query, engine)
df_stops

,stop_id,stop_name,stop_lat,stop_lon
0,6829,EB 3 Street SW Station (Free Fare Zone),51.046606,-114.068904
1,3631,EB 45 Street CTrain Station,51.037868,-114.153580
2,6828,EB 6 Street SW Station (Free Fare Zone),51.046765,-114.074858
3,3627,EB 69 Street CTrain Station,51.037507,-114.188475
4,6827,EB 8 Street SW Station (Free Fare Zone),51.046916,-114.079953
...,...,...,...,...
78,3634,WB Shaganappi Point CTrain Station,51.041603,-114.124256
79,3628,WB Sirocco CTrain Station,51.038152,-114.169585
80,3636,WB Sunalta CTrain Station,51.044856,-114.100198
81,3632,WB Westbrook CTrain Station,51.040547,-114.136596


In [31]:
free_fare_query = """
WITH scheduled_arrivals AS (
    SELECT 
        s.stop_id,
        s.stop_name,
        r.route_short_name,
        st.arrival_sec,
        st.arrival_sec - LAG(st.arrival_sec) OVER (
            PARTITION BY st.stop_id, r.route_short_name 
            ORDER BY st.arrival_sec
        ) AS scheduled_headway_sec
    FROM stop_times st
    JOIN trips t ON st.trip_id = t.trip_id
    JOIN routes r ON t.route_id = r.route_id
    JOIN stops s ON st.stop_id = s.stop_id
    WHERE s.stop_name LIKE '%Free Fare Zone%' OR s.stop_name LIKE '%Free Fare%'
)
SELECT 
    stop_name,
    route_short_name,
    COUNT(*) AS total_scheduled_trips,
    ROUND(AVG(scheduled_headway_sec) / 60.0, 2) AS avg_headway_min,
    ROUND(MIN(scheduled_headway_sec) / 60.0, 2) AS min_headway_min,
    ROUND(MAX(scheduled_headway_sec) / 60.0, 2) AS max_headway_min
FROM scheduled_arrivals
WHERE scheduled_headway_sec IS NOT NULL
GROUP BY stop_name, route_short_name
ORDER BY stop_name, route_short_name;
"""

df_free_fare = pd.read_sql_query(free_fare_query, engine)
df_free_fare

,stop_name,route_short_name,total_scheduled_trips,avg_headway_min,min_headway_min,max_headway_min
0,EB 3 Street SW Station (Free Fare Zone),201,579,2.19,0.0,16.0
1,EB 3 Street SW Station (Free Fare Zone),202,556,2.27,0.0,15.0
2,EB 6 Street SW Station (Free Fare Zone),201,579,2.19,0.0,15.0
3,EB 6 Street SW Station (Free Fare Zone),202,556,2.27,0.0,15.0
4,EB 8 Street SW Station (Free Fare Zone),201,904,1.44,0.0,12.0
5,EB 8 Street SW Station (Free Fare Zone),202,556,2.27,0.0,15.0
6,EB Centre Street Station (Free Fare Zone),201,579,2.19,0.0,15.0
7,EB Centre Street Station (Free Fare Zone),202,556,2.27,0.0,15.0
8,EB City Hall/Bow Valley College (Free Fare Zone),201,1365,0.96,0.0,8.0
9,EB City Hall/Bow Valley College (Free Fare Zone),202,556,2.27,0.0,15.0


In [12]:
raw_headways_query = """
SELECT 
    s.stop_name,
    r.route_short_name,
    st.arrival_sec,
    (st.arrival_sec - LAG(st.arrival_sec) OVER (
        PARTITION BY st.stop_id, r.route_short_name 
        ORDER BY st.arrival_sec
    )) / 60.0 AS headway_min
FROM stop_times st
JOIN trips t ON st.trip_id = t.trip_id
JOIN routes r ON t.route_id = r.route_id
JOIN stops s ON st.stop_id = s.stop_id
WHERE s.stop_name LIKE '%Free Fare Zone%' OR s.stop_name LIKE '%Free Fare%'
"""

df_raw = pd.read_sql_query(raw_headways_query, engine).dropna()


def calculate_swt(headways):
    if len(headways) == 0 or sum(headways) == 0:
        return 0
    return sum(headways**2) / (2 * sum(headways))


results = []
for (stop, route), group in df_raw.groupby(["stop_name", "route_short_name"]):
    headways = group["headway_min"].values
    mean_h = np.mean(headways)
    swt = calculate_swt(headways)
    results.append(
        {
            "Stop Name": stop,
            "Route": route,
            "Mean Headway (min)": round(mean_h, 2),
            "Scheduled Wait Time (SWT min)": round(swt, 2),
            "Variance Penalty (min)": round(swt - (mean_h / 2), 2),
        }
    )

df_ewt_base = pd.DataFrame(results)
df_ewt_base

,Stop Name,Route,Mean Headway (min),Scheduled Wait Time (SWT min),Variance Penalty (min)
0,EB 3 Street SW Station (Free Fare Zone),201,2.19,3.16,2.06
1,EB 3 Street SW Station (Free Fare Zone),202,2.27,3.40,2.27
2,EB 6 Street SW Station (Free Fare Zone),201,2.19,3.04,1.95
3,EB 6 Street SW Station (Free Fare Zone),202,2.27,3.34,2.20
4,EB 8 Street SW Station (Free Fare Zone),201,1.44,2.62,1.90
5,EB 8 Street SW Station (Free Fare Zone),202,2.27,3.40,2.27
6,EB Centre Street Station (Free Fare Zone),201,2.19,3.04,1.95
7,EB Centre Street Station (Free Fare Zone),202,2.27,3.34,2.20
8,EB City Hall/Bow Valley College (Free Fare Zone),201,0.96,2.10,1.62
9,EB City Hall/Bow Valley College (Free Fare Zone),202,2.27,3.34,2.20


In [33]:
shapes_df = pd.read_sql_query(
    "SELECT shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence FROM shapes ORDER BY shape_id, CAST(shape_pt_sequence AS INTEGER)",
    engine,
)
stops_df = pd.read_sql_query(
    "SELECT stop_id, stop_name, stop_lat, stop_lon FROM stops", engine
)

stops_summary = pd.merge(stops_df, df_ewt_base, left_on="stop_name", right_on="Stop Name", how="left")
stops_summary

,stop_id,stop_name,stop_lat,stop_lon,Stop Name,Route,Mean Headway (min),Scheduled Wait Time (SWT min),Variance Penalty (min)
0,3626,WB 69 Street CTrain Station,51.037605,-114.188486,NaN,NaN,NaN,NaN,NaN
1,3627,EB 69 Street CTrain Station,51.037507,-114.188475,NaN,NaN,NaN,NaN,NaN
2,3628,WB Sirocco CTrain Station,51.038152,-114.169585,NaN,NaN,NaN,NaN,NaN
3,3629,EB Sirocco CTrain Station,51.038105,-114.169484,NaN,NaN,NaN,NaN,NaN
4,3630,WB 45 Street CTrain Station,51.037942,-114.153369,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
87,9396,NB Martindale CTrain Station,51.117409,-113.967973,NaN,NaN,NaN,NaN,NaN
88,9398,NB Saddletowne CTrain Station,51.125542,-113.948277,NaN,NaN,NaN,NaN,NaN
89,9781,SB Saddletowne CTrain Station,51.125895,-113.948358,NaN,NaN,NaN,NaN,NaN
90,9896,NB McKnight - Westwinds CTrain Station,51.109530,-113.975160,NaN,NaN,NaN,NaN,NaN


In [34]:
lines = []
for shape_id, group in shapes_df.groupby("shape_id"):
    if len(group) > 1:
        points = [
            Point(float(lon), float(lat))
            for lat, lon in zip(group["shape_pt_lat"], group["shape_pt_lon"])
        ]
        lines.append({"shape_id": shape_id, "geometry": LineString(points)})

gdf_lines = gpd.GeoDataFrame(lines, crs="EPSG:4326")

In [35]:
calgary_map = folium.Map(
    location=[51.046, -114.071], zoom_start=14, tiles="cartodbpositron"
)

In [36]:
for _, row in gdf_lines.iterrows():
    sim_geo = gpd.GeoSeries(row["geometry"])
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(
        data=geo_j, style_function=lambda x: {"color": "#003366", "weight": 3, "opacity": 0.6}
    )
    geo_j.add_to(calgary_map)

In [37]:
for _, row in stops_summary.iterrows():
    lat, lon = float(row["stop_lat"]), float(row["stop_lon"])
    is_free_fare = "Free Fare" in str(row["stop_name"])

    marker_color = "red" if is_free_fare else "blue"
    radius = 7 if is_free_fare else 4

    popup_text = f"""
    <b>{row['stop_name']}</b><br>
    <b>Route:</b> {row.get('Route', 'CTrain System')}<br>
    <b>Mean Headway:</b> {row.get('Mean Headway (min)', 'N/A')} min<br>
    <b>Scheduled Wait Time (SWT):</b> {row.get('Scheduled Wait Time (SWT min)', 'N/A')} min<br>
    <b>Variance Penalty:</b> {row.get('Variance Penalty (min)', 'N/A')} min
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        popup=folium.Popup(popup_text, max_width=300),
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.8,
    ).add_to(calgary_map)

calgary_map.save("data/ctrain_free_fare_map.html")
calgary_map